### **Assignment** **2**
##### Name :Priya Rajkumnar Jadhav
##### Class :TY-B
##### Batch :B
##### Roll No: 23107098

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import tensorflow as tf

dataset_path = "/content/drive/MyDrive/College/SEM-6/AIPD/Dataset/covid_1000/images"

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    labels=None,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    labels=None,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)


Found 1000 files.
Using 800 files for training.
Found 1000 files.
Using 200 files for validation.


In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x: normalization_layer(x))
val_ds   = val_ds.map(lambda x: normalization_layer(x))


In [ ]:
train_ds_ae = train_ds.map(lambda x: (x, x))
val_ds_ae   = val_ds.map(lambda x: (x, x))


In [ ]:
from tensorflow.keras import layers, models

input_img = layers.Input(shape=(224, 224, 3))

# Encoder
x = layers.Conv2D(32, 3, activation='relu', padding='same')(input_img)
x = layers.MaxPooling2D(2, padding='same')(x)

x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D(2, padding='same')(x)

x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
encoded = layers.MaxPooling2D(2, padding='same')(x)

# Decoder
x = layers.Conv2D(128, 3, activation='relu', padding='same')(encoded)
x = layers.UpSampling2D(2)(x)

x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.UpSampling2D(2)(x)

x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = layers.UpSampling2D(2)(x)

decoded = layers.Conv2D(3, 3, activation='sigmoid', padding='same')(x)

model = models.Model(input_img, decoded)
model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 28, 28, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_3 (UpSampling2D)  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 56, 56, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_4 (UpSampling2D)  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 112, 112, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_5 (UpSampling2D)  │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 224, 224, 3)    │           867 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333,955 (1.27 MB)

 Trainable params: 333,955 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='mse'
)


In [ ]:
history = model.fit(
    train_ds_ae,
    validation_data=val_ds_ae,
    epochs=10
)


Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 292s 11s/step - loss: 0.0456 - val_loss: 0.0157
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 292s 10s/step - loss: 0.0134 - val_loss: 0.0053
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 272s 11s/step - loss: 0.0043 - val_loss: 0.0031
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 264s 11s/step - loss: 0.0027 - val_loss: 0.0023
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 344s 11s/step - loss: 0.0022 - val_loss: 0.0020
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 275s 11s/step - loss: 0.0018 - val_loss: 0.0033
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 269s 11s/step - loss: 0.0022 - val_loss: 0.0014
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 270s 11s/step - loss: 0.0013 - val_loss: 0.0013
Epoch 9/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 264s 11s/step - loss: 0.0013 - val_loss: 0.0011
Epoch 10/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 272s 11s/step - loss: 0.0012 - val_loss: 0.0012
